# r–ρ–MI case study — channel-space explainer + run processing (2026-06-22)

Phase portraits, a $\beta$-sweep, per-pair inspection helpers reading the real pyspi outputs,
SNR-regime comparison, SPI–SPI planes with the meta-feature $f_{ij}=\mathrm{corr}(\mathrm{SPI}_i,\mathrm{SPI}_j)$,
and processing of the `260622_g-roll` run.

**Construction.** Shared AR(1) mother $z$ (standardised). Each channel = filter($z$), standardised
to unit variance, + i.i.d. noise ($\text{SNR}=1/\text{noise\_std}^2$; `ar1_a`=smoothness). Filters:
**L** $=z$ | **M** $=$ sigmoid($\beta z$) | **NM** $=(z-\bar z)^2$. Generator
`src.generators.generate_filter_roll_mts`; config `configs/generate/r_rho_mi/260622_g-roll.yaml`.

In [ ]:
import numpy as np
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt

def ar1_mother(T, a=0.8, rng=None):
    rng = rng or np.random.default_rng(0)
    m = np.zeros(T)
    for t in range(1, T):
        m[t] = a * m[t-1] + rng.normal(0, 1)
    return (m - m.mean()) / m.std()

def unit(g):
    return (g - g.mean()) / g.std()

def sigmoid(z, beta):           # monotone-nonlinear filter, gain beta
    return 1.0 / (1.0 + np.exp(-beta * z))

def mi(a, b):                   # sklearn KNN MI proxy (synthetic cells only)
    return float(mutual_info_regression(a.reshape(-1, 1), b, n_neighbors=4, random_state=0)[0])

T = 2000
z = ar1_mother(T, a=0.8, rng=np.random.default_rng(4))
beta = 2.5                      # <-- tweak me
L  = unit(z)
M  = unit(sigmoid(z, beta))
NM = unit((z - z.mean())**2)

## Channel–channel spaces (synthetic): pair-type × SNR

In [ ]:
pairs = {"L-L": (L, L), "M-M": (M, M), "NM-NM": (NM, NM),
         "L-M": (L, M), "L-NM": (L, NM), "M-NM": (M, NM)}
noise = [0.1, 0.3, 0.6, 1.0, 1.6]          # noise_std on unit-variance signal; SNR = 1/std^2

rng = np.random.default_rng(1)
nr, nc = len(pairs), len(noise)
fig, axes = plt.subplots(nr, nc, figsize=(2.5*nc, 2.5*nr))
for i, (name, (a0, b0)) in enumerate(pairs.items()):
    for j, ns in enumerate(noise):
        xi, xj = a0 + rng.normal(0, ns, T), b0 + rng.normal(0, ns, T)
        rr, rho, mm = np.corrcoef(xi, xj)[0, 1], spearmanr(xi, xj)[0], mi(xi, xj)
        ax = axes[i, j]
        ax.scatter(xi, xj, s=2, alpha=0.15, color="navy", linewidths=0)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title((f"SNR={1/ns**2:.0f}\n" if i == 0 else "") + f"r={rr:+.2f} rho={rho:+.2f} MI={mm:.2f}", fontsize=7)
        if j == 0:
            ax.set_ylabel(name, fontweight="bold", fontsize=11)
fig.suptitle(f"Channel-channel space (sigmoid beta={beta}): pair-type (rows) x SNR (cols)", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

## Inspect real datasets: per-pair phase portrait + raw SPI bars

`plot_pair_phase` / `plot_pair_spis` read the actual pyspi outputs (`spi_mpis.npz`) + `timeseries.npy`; `plot_spi_plane` scatters every pair in an SPI–SPI plane and annotates the meta-feature $f_{ij}$. All **return an `ax`**. `find_pair(path, 'L', 'M', snr='high'|'median'|'low')` selects a representative pair by type and SNR (from `meta.json` `noise_stds`). MI bars use the **Linfoot** correlation `√(1−e^{−2I})` so they share the r/ρ scale.

In [ ]:
import json
from pathlib import Path
from src.utils import project_root

def _resolve_spi_keys(files):
    km = {}
    for k in files:
        kl = k.lower()
        if "cov" in kl or "empirical" in kl: km["r"] = k
        elif "spearman" in kl: km["rho"] = k
        elif "mi_" in kl or "kraskov" in kl or "mutual" in kl: km["MI"] = k
    return km

def load_dataset(dataset_path):
    """Load one run dataset -> (timeseries (T,M), mpis {r,rho,MI:(M,M)}, gen-meta dict)."""
    p = Path(dataset_path)
    ts = np.load(p / "timeseries.npy")
    npz = np.load(p / "spi_mpis.npz")
    km = _resolve_spi_keys(npz.files)
    mpis = {k: npz[km[k]] for k in ("r", "rho", "MI")}
    meta = json.loads((p / "meta.json").read_text())["generator"] if (p / "meta.json").exists() else {}
    return ts, mpis, meta

_ABBR = {"linear": "L", "monotonic": "M", "non-monotonic": "NM"}
_MIXED_COLORS = {"L-M": "#0072B2", "L-NM": "#D55E00", "M-NM": "#009E73"}

def linfoot(mi_nats):
    """MI (nats) -> Linfoot informational correlation sqrt(1-exp(-2I)) in [0,1].
    Equals |r| for a bivariate Gaussian, so it puts Kraskov MI on the r/rho scale."""
    return float(np.sqrt(np.clip(1.0 - np.exp(-2.0 * max(mi_nats, 0.0)), 0.0, 1.0)))

def find_pair(dataset_path, type_x, type_y, snr="high"):
    """Channel pair (i<j) matching {type_x,type_y} (use 'L','M','NM'). Among matches, snr picks by
    combined per-channel noise (meta.json noise_stds): 'high'=lowest noise, 'low'=highest, 'median'."""
    _, _, meta = load_dataset(dataset_path)
    types, noise = meta.get("types"), meta.get("noise_stds")
    want = {type_x, type_y}
    cands = [(i, j) for i in range(len(types)) for j in range(i + 1, len(types))
             if {_ABBR[types[i]], _ABBR[types[j]]} == want]
    if not cands:
        raise ValueError(f"no {type_x}-{type_y} pair in {dataset_path}")
    if noise is not None:
        cands.sort(key=lambda ij: noise[ij[0]] + noise[ij[1]])      # ascending -> high SNR first
        return {"high": cands[0], "low": cands[-1], "median": cands[len(cands)//2]}.get(snr, cands[0])
    return cands[0]

def plot_pair_phase(dataset_path, channel_x, channel_y, ax=None, s=4, alpha=0.3):
    """Phase portrait X[:,x] vs X[:,y] (timeseries.npy), annotated with the dataset's actual pyspi
    r/rho/MI for that pair (spi_mpis.npz). Returns ax."""
    if ax is None:
        _, ax = plt.subplots(figsize=(3.2, 3.2))
    ts, mpis, meta = load_dataset(dataset_path)
    ax.scatter(ts[:, channel_x], ts[:, channel_y], s=s, alpha=alpha, color="navy", linewidths=0)
    r, rho, mi_ = (mpis[k][channel_x, channel_y] for k in ("r", "rho", "MI"))
    types = meta.get("types")
    tag = f"{_ABBR[types[channel_x]]}-{_ABBR[types[channel_y]]}  " if types else ""
    ax.set_title(f"{tag}ch {channel_x},{channel_y}\nr={r:+.2f}  rho={rho:+.2f}  MI={mi_:.2f}", fontsize=8)
    ax.set_xlabel(f"X[{channel_x}]"); ax.set_ylabel(f"X[{channel_y}]"); ax.set_xticks([]); ax.set_yticks([])
    return ax

def plot_pair_spis(dataset_path, channel_x, channel_y, ax=None, mi_scale="linfoot"):
    """Bar plot of r/rho/MI for one channel pair. mi_scale='linfoot' shows MI as the Linfoot
    informational correlation in [0,1] (comparable to |r|,|rho|); 'raw' shows MI in nats. Returns ax."""
    if ax is None:
        _, ax = plt.subplots(figsize=(2.8, 3.2))
    _, mpis, _ = load_dataset(dataset_path)
    r = mpis["r"][channel_x, channel_y]; rho = mpis["rho"][channel_x, channel_y]
    mi_raw = mpis["MI"][channel_x, channel_y]
    mi_show = linfoot(mi_raw) if mi_scale == "linfoot" else mi_raw
    mi_lab = "MI->r" if mi_scale == "linfoot" else "MI"
    vals = [r, rho, mi_show]
    bars = ax.bar(["r", "rho", mi_lab], vals, color=["#4C72B0", "#DD8452", "#55A868"], width=0.65)
    ax.axhline(0, color="grey", lw=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v, f"{v:.2f}", ha="center",
                va="bottom" if v >= 0 else "top", fontsize=8)
    top = 1.05 if mi_scale == "linfoot" else max(1.0, max(vals)) + 0.18
    ax.set_ylim(min(0, r, rho) - 0.12, top)
    ax.set_ylabel("SPI value" + (" (MI: Linfoot r)" if mi_scale == "linfoot" else " (MI: nats)"))
    ax.grid(True, axis="y", alpha=0.3)
    return ax

def plot_spi_plane(dataset_path, spi_x, spi_y, ax=None, annotate=True):
    """Scatter MPI pairs in the (spi_x, spi_y) plane (spi in {'r','rho','MI'}). Same-type (X-X)
    pairs are grey/hollow (context); mixed-type (X-Y) pairs coloured by relation (signal). Annotates
    f_{ij}=corr(SPI_x,SPI_y) over ALL pairs (raw values -> matches the run pipeline). Returns ax."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))
    _, mpis, meta = load_dataset(dataset_path)
    types = meta["types"]; iu = np.triu_indices(len(types), 1)
    xv, yv = mpis[spi_x][iu], mpis[spi_y][iu]
    labs = np.array(["-".join(sorted([_ABBR[types[i]], _ABBR[types[j]]])) for i, j in zip(*iu)])
    same = np.array([l.split("-")[0] == l.split("-")[1] for l in labs])
    ax.scatter(xv[same], yv[same], s=26, facecolors="none", edgecolors="#999999",
               linewidths=0.8, alpha=0.6, label="same-type (X-X)")
    for lab, col in _MIXED_COLORS.items():
        m = labs == lab
        if m.any():
            ax.scatter(xv[m], yv[m], s=36, color=col, alpha=0.85, label=lab)
    f = float(np.corrcoef(xv, yv)[0, 1])
    if annotate:
        ax.text(0.04, 0.96, f"$f_{{{spi_x},{spi_y}}}$ = {f:+.3f}", transform=ax.transAxes,
                va="top", fontsize=11, bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    ax.set_xlabel(spi_x); ax.set_ylabel(spi_y); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")
    return ax

In [ ]:
# Per-pair mechanism: phase portrait (top) + r/rho/MI bars (bottom; MI on Linfoot correlation
# scale) for representative pair-types in one iter3 dataset. Helpers return an ax -> drop into a grid.
ds = project_root() / "data/r_rho_mi/260622_g-roll/iter3_LMNM/M20_T2000_I0"
pair_types = [("L", "L"), ("L", "M"), ("L", "NM"), ("M", "NM")]

fig, axes = plt.subplots(2, len(pair_types), figsize=(3.0 * len(pair_types), 6))
for c, (tx, ty) in enumerate(pair_types):
    i, j = find_pair(ds, tx, ty, snr="high")
    plot_pair_phase(ds, i, j, ax=axes[0, c])
    plot_pair_spis(ds, i, j, ax=axes[1, c])
fig.suptitle("Per-pair mechanism: phase portrait (top) + r/rho/MI bars (bottom, MI as Linfoot r)", y=1.0)
fig.tight_layout(); plt.show()

## SNR regimes on one signature (within-MTS)

SNR is *not* varied across datasets — each MTS has a per-channel noise spread, so `find_pair(..., snr=...)` picks a high/median/low-SNR pair of a fixed type. A true dataset-level SNR sweep would need a `noise_std` grid in the config (easy to add).

In [ ]:
# SNR regimes on one signature: the SAME pair-type (L-M) at high / median / low within-MTS SNR,
# selected via find_pair(..., snr=...). Shows the signature strength scaling with SNR.
ds = project_root() / "data/r_rho_mi/260622_g-roll/iter2_LM/M20_T2000_I0"
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for c, regime in enumerate(["high", "median", "low"]):
    i, j = find_pair(ds, "L", "M", snr=regime)
    plot_pair_phase(ds, i, j, ax=axes[0, c])
    axes[0, c].set_title(f"L-M | {regime} SNR\n" + axes[0, c].get_title(), fontsize=8)
    plot_pair_spis(ds, i, j, ax=axes[1, c])
fig.suptitle("Same pair-type (L-M), 3 within-MTS SNR regimes: weaker SNR -> all of r/rho/MI attenuate together")
fig.tight_layout(); plt.show()

## β-sweep (single L–M pair): how far does monotone nonlinearity break {r, ρ}?

In [ ]:
# Can monotone nonlinearity break {r, rho}?  Sweep sigmoid gain beta for an L-M pair.
betas = [0.5, 1, 2, 3, 4, 6, 8, 12, 20]
rng = np.random.default_rng(2)
rows = []
for b in betas:
    Mb = unit(sigmoid(z, b))
    xi, xj = unit(z) + rng.normal(0, 0.05, T), Mb + rng.normal(0, 0.05, T)
    rows.append((b, np.corrcoef(xi, xj)[0, 1], spearmanr(xi, xj)[0], mi(xi, xj)))
rows = np.array(rows)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="r (Pearson)")
ax.plot(rows[:, 0], rows[:, 2], "s-", label="rho (Spearman)")
ax.axhline(np.sqrt(2/np.pi), ls="--", color="grey", lw=0.8, label="sqrt(2/pi) = corr(z, sign z)")
ax.set_xlabel("sigmoid gain beta"); ax.set_ylabel("correlation"); ax.set_xscale("log")
ax.set_title("L-M pair: rho ~1 then falls as beta->step (ties); r floors ~0.80")
ax.legend(); fig.tight_layout(); plt.show()

## MTS-level β: which β drops the *feature* corr(r, ρ)?

The MTS-level **feature** `corr(r,ρ)` falls *monotonically* with β while `corr(ρ,MI)` stays ~0.92 (real pyspi: π→0.88, 4→0.83, 2π→0.78). The config uses β=4.

In [ ]:
from src.generators import generate_filter_roll_mts

def _spi_corrs(X):
    R = np.corrcoef(X, rowvar=False); S = spearmanr(X)[0]
    Mn = X.shape[1]; I = np.zeros((Mn, Mn))
    for i in range(Mn):
        for j in range(i + 1, Mn):
            I[i, j] = I[j, i] = mutual_info_regression(X[:, [i]], X[:, j], n_neighbors=4, random_state=0)[0]
    iu = np.triu_indices(Mn, 1)
    fc = lambda a, b: np.corrcoef(a, b)[0, 1]
    return fc(R[iu], S[iu]), fc(S[iu], I[iu])

betas = [1, 2, np.pi, 4, 2*np.pi, 8]
rows = []
for b in betas:
    X = generate_filter_roll_mts(M=20, T=2000, n_linear=10, n_monotonic=10, beta=b,
                                 noise_std=0.1, noise_std_variable=True, noise_std_scale=2.0,
                                 ar1_a=0.8, zscore=True, rng=np.random.default_rng(100))
    rows.append((b, *_spi_corrs(X)))
rows = np.array(rows)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="corr(r, rho)  <- want this to drop")
ax.plot(rows[:, 0], rows[:, 2], "s-", label="corr(rho, MI)  <- want this high")
for x in (np.pi, 2*np.pi): ax.axvline(x, ls=":", color="grey", lw=0.8)
ax.set_xscale("log"); ax.set_xlabel("sigmoid gain beta"); ax.set_ylabel("meta-feature corr")
ax.set_title("iter-2 (L+M): corr(r,rho) drops with beta; corr(rho,MI) stays high")
ax.legend(); fig.tight_layout(); plt.show()
print(rows)

## Process the run: staggered meta-feature table (grouped bars)

Aggregate `corr(SPI_i, SPI_j)` per iteration (mean ± std). **Grouped bars, not a line** — iterations are discrete conditions, not a swept continuum. (A 3×3 annotated heatmap of the same numbers is the most compact paper panel.)

In [ ]:
BASE = project_root() / "data/r_rho_mi/260622_g-roll"
ITERS = ["iter1_L", "iter2_LM", "iter3_LMNM"]
FEATS = ["corr(r,rho)", "corr(r,MI)", "corr(rho,MI)"]

def _offdiag(Mm):
    iu = np.triu_indices(Mm.shape[0], 1); return Mm[iu]

def staggered_stats(base, iters=ITERS):
    means, stds = {p: [] for p in FEATS}, {p: [] for p in FEATS}
    for it in iters:
        per = []
        for d in sorted((Path(base) / it).glob("M*_T*_I*")):
            if not (d / "spi_mpis.npz").exists():
                continue
            _, mpis, _ = load_dataset(d)
            rv, sv, iv = _offdiag(mpis["r"]), _offdiag(mpis["rho"]), _offdiag(mpis["MI"])
            fc = lambda a, b: float(np.corrcoef(a, b)[0, 1])
            per.append({"corr(r,rho)": fc(rv, sv), "corr(r,MI)": fc(rv, iv), "corr(rho,MI)": fc(sv, iv)})
        for p in FEATS:
            v = np.array([f[p] for f in per]); means[p].append(v.mean()); stds[p].append(v.std())
    return means, stds

def plot_staggered(base, iters=ITERS, ax=None):
    """Grouped bars: 3 meta-features per iteration (mean +/- std). Returns ax.
    Bars not lines: iterations are discrete conditions, not a swept continuum."""
    means, stds = staggered_stats(base, iters)
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.5))
    x = np.arange(len(iters)); w = 0.26
    for k, p in enumerate(FEATS):
        ax.bar(x + (k - 1) * w, means[p], w, yerr=stds[p], capsize=3, label=p,
               color=["#4C72B0", "#DD8452", "#55A868"][k])
    ax.set_xticks(x); ax.set_xticklabels(iters); ax.set_ylim(0, 1.08)
    ax.set_ylabel("meta-feature  corr(SPI_i, SPI_j)")
    ax.set_title("Staggered decoupling: corr(r,rho) drops at iter2; corr(.,MI) drops at iter3")
    ax.legend(fontsize=8, loc="lower left"); ax.grid(True, axis="y", alpha=0.3)
    return ax

plot_staggered(BASE); plt.show()
means, stds = staggered_stats(BASE)
print(f"{'iter':>12} " + " ".join(f"{p:>16}" for p in FEATS))
for i, it in enumerate(ITERS):
    print(f"{it:>12} " + " ".join(f"{means[p][i]:7.3f}+/-{stds[p][i]:.3f}" for p in FEATS))

## SPI–SPI planes with the meta-feature $f_{ij}$

Same-type (X–X) pairs are grey/hollow context; mixed-type (X–Y) pairs are coloured signal. Each plane is annotated with $f_{ij}=\mathrm{corr}(\mathrm{SPI}_i,\mathrm{SPI}_j)$ over **all** pairs (raw values, matching the pipeline).

In [ ]:
# SPI-SPI planes for one dataset: f_{ij} = corr(SPI_i, SPI_j) annotated per plane.
# Same-type pairs grey/hollow (context); mixed-type pairs coloured (the signal).
ds = project_root() / "data/r_rho_mi/260622_g-roll/iter3_LMNM/M20_T2000_I0"
planes = [("r", "rho"), ("r", "MI"), ("rho", "MI")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (sx, sy) in zip(axes, planes):
    plot_spi_plane(ds, sx, sy, ax=ax)
fig.suptitle("iter3 SPI-SPI planes: mixed-type pairs (coloured) carry the decoupling; f = corr over all pairs", y=1.02)
fig.tight_layout(); plt.show()